In [1]:
import pandas as pd

In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\krise\Vanatara\raw_data.csv",encoding="latin1")
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
df.shapec

(180519, 53)

# Drop sensitive / irrelevant columns

In [4]:
cols_to_drop = ['Customer Email', 'Customer Password', 'Customer Fname',
                 'Customer Lname', 'Customer Street', 'Product Image',
                 'Product Description', 'Order Zipcode', 'Customer Zipcode']

In [5]:
df = df.drop(columns=cols_to_drop)

In [6]:
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])
df = df.rename(columns={
    'order date (DateOrders)': 'order_date',
    'shipping date (DateOrders)': 'shipping_date'})

In [7]:
text_cols = ['Customer City', 'Customer State', 'Customer Country',
             'Order City', 'Order Country', 'Category Name',
             'Department Name', 'Product Name']

In [8]:
for col in text_cols:
    df[col] = df[col].astype(str).str.title()

In [9]:
df['fulfilment_time_days'] = (df['shipping_date'] - df['order_date']).dt.days

# Build Location dimension FIRST (with surrogate key)
# then merge back into df so fact table can use it

In [10]:
dim_location = df[['Order City', 'Order Country', 'Market', 'Order Region',
                    'Order State']].drop_duplicates().reset_index(drop=True)

In [11]:
dim_location['location_id'] = dim_location.index + 1

In [12]:
df = df.merge(dim_location, on=['Order City', 'Order Country', 'Market',
                                  'Order Region', 'Order State'], how='left')

In [13]:
dim_customers = df[['Order Customer Id', 'Customer Segment', 'Customer City',
                     'Customer State', 'Customer Country']].drop_duplicates(subset=['Order Customer Id'])


In [14]:
dim_products = df[['Order Item Cardprod Id', 'Category Name', 'Department Name',
                    'Product Name', 'Product Price']].drop_duplicates(subset=['Order Item Cardprod Id'])


# Date table

In [15]:
date_range = pd.date_range(start=df['order_date'].min(), end=df['order_date'].max(), freq='D')
dim_date = pd.DataFrame({'date': date_range})
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%b')
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['day_of_week'] = dim_date['date'].dt.day_name()

# Fact table

In [16]:
fact_orders = df[['Order Id', 'Order Item Id', 'order_date', 'shipping_date',
                   'fulfilment_time_days', 'Order Customer Id', 'Order Item Cardprod Id',
                   'location_id', 'Sales', 'Order Item Total', 'Order Profit Per Order',
                   'Order Item Discount', 'Order Item Discount Rate',
                   'Order Item Quantity', 'Order Item Profit Ratio',
                   'Shipping Mode', 'Delivery Status', 'Late_delivery_risk',
                   'Order Status', 'Type']].copy()

In [17]:
fact_orders = fact_orders.rename(columns={
    'Order Id': 'order_id', 'Order Item Id': 'order_item_id',
    'Order Customer Id': 'customer_id', 'Order Item Cardprod Id': 'product_id',
    'Sales': 'sales', 'Order Item Total': 'order_item_total',
    'Order Profit Per Order': 'order_profit_per_order',
    'Order Item Discount': 'order_item_discount',
    'Order Item Discount Rate': 'order_item_discount_rate',
    'Order Item Quantity': 'order_item_quantity',
    'Order Item Profit Ratio': 'order_item_profit_ratio',
    'Shipping Mode': 'shipping_mode', 'Delivery Status': 'delivery_status',
    'Late_delivery_risk': 'late_delivery_risk', 'Order Status': 'order_status',
    'Type': 'payment_type'
})

# dim_customers

In [18]:
dim_customers = dim_customers.rename(columns={
    'Order Customer Id': 'customer_id', 'Customer Segment': 'customer_segment',
    'Customer City': 'customer_city', 'Customer State': 'customer_state',
    'Customer Country': 'customer_country'
})

# dim_products

In [19]:
dim_products = dim_products.rename(columns={
    'Order Item Cardprod Id': 'product_id', 'Category Name': 'category_name',
    'Department Name': 'department_name', 'Product Name': 'product_name',
    'Product Price': 'product_price'})

# dim_location

In [22]:
dim_location = dim_location.rename(columns={
    'Order City': 'order_city', 'Order Country': 'order_country',
    'Market': 'market', 'Order Region': 'order_region', 'Order State': 'order_state'
})

dim_location = dim_location[['location_id', 'order_city', 'order_country', 
                              'market', 'order_region', 'order_state']]
dim_location.to_csv('dim_location.csv', index=False)

In [23]:
print("fact_orders:", fact_orders.shape)
print("dim_customers:", dim_customers.shape)
print("dim_products:", dim_products.shape)
print("dim_location:", dim_location.shape)
print("dim_date:", dim_date.shape)

fact_orders: (180519, 20)
dim_customers: (20652, 5)
dim_products: (118, 5)
dim_location: (3772, 6)
dim_date: (1127, 6)


In [24]:
print("\nnulls in fact_orders.location_id:", fact_orders['location_id'].isnull().sum())
print("nulls overall in fact_orders:\n", fact_orders.isnull().sum()[fact_orders.isnull().sum() > 0])
print("\norder_item_id duplicates:", fact_orders['order_item_id'].duplicated().sum())


nulls in fact_orders.location_id: 0
nulls overall in fact_orders:
 Series([], dtype: int64)

order_item_id duplicates: 0


In [25]:
missing_cust = ~fact_orders['customer_id'].isin(dim_customers['customer_id'])
missing_prod = ~fact_orders['product_id'].isin(dim_products['product_id'])
print("orders with missing customer link:", missing_cust.sum())
print("orders with missing product link:", missing_prod.sum())

orders with missing customer link: 0
orders with missing product link: 0


In [28]:
fact_orders.to_csv('fact_orders.csv', index=False)
dim_customers.to_csv('dim_customers.csv', index=False)
dim_products.to_csv('dim_products.csv', index=False)
dim_location.to_csv('dim_location.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)